In [1]:
import os
import pickle
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.pool import QueuePool

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location


def fetch_and_save_data(query, filename_suffix, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        with engine.connect() as conn:
            # Execute the query
            df = pd.read_sql_query(query, conn, params=params)

        print(df)
        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
    SELECT ds.id AS source_id, COUNT(dls.domain_list_id) AS domain_count, ds.organisation
    FROM DomainListDomainSource dls
    JOIN DomainSource ds ON dls.domain_source_id = ds.id
    GROUP BY ds.id, ds.organisation
    """
    filename_suffix = "domain_source_counts"
    fetch_and_save_data(query, filename_suffix)

Error: (psycopg2.errors.UndefinedTable) relation "domainlistdomainsource" does not exist
LINE 3:     FROM DomainListDomainSource dls
                 ^

[SQL: 
    SELECT ds.id AS source_id, COUNT(dls.domain_list_id) AS domain_count, ds.organisation
    FROM DomainListDomainSource dls
    JOIN DomainSource ds ON dls.domain_source_id = ds.id
    GROUP BY ds.id, ds.organisation
    ]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [2]:
# Function to retrieve total number of domains formatted with dots
def get_total_domains_formatted(cursor):
    query = "SELECT COUNT(*) AS total_domains FROM DomainList"
    try:
        cursor.execute(query)
        result = cursor.fetchone()
        total_domains = result["total_domains"]
        formatted_total_domains = f"{total_domains:,}".replace(",", ".")
        print(f"Total number of domains in the database: {formatted_total_domains}")
    except psycopg2.Error as e:  # Changed to psycopg2.Error
        print(f"Error executing query: {e}")


# Example usage with the cursor obtained from connect_to_db()
get_total_domains_formatted(cursor)

NameError: name 'cursor' is not defined

In [ ]:
# Query to fetch TLD data
query = """
    SELECT tld, COUNT(*) AS count
    FROM DomainList
    GROUP BY tld
    ORDER BY count DESC
    """

# Execute the query and fetch the data
cursor.execute(query)
tld_data = cursor.fetchall()

# Create a pandas DataFrame from the fetched data
df = pd.DataFrame(tld_data)  # No need for columns argument, DictCursor handles it

# Keep the 'count' column as integers for calculations
df["count"] = df["count"].astype(int)

# Calculate the relative amount of each TLD
total_domains = df["count"].sum()
df["relative"] = (df["count"] / total_domains * 100).map("{:.2f}%".format)  # Format directly

# Format the 'count' column with thousand separators for display
df["count"] = df["count"].apply(lambda x: f"{x:,}".replace(",", "."))

# Display the top 25 TLDs by count
print("Top 25 TLDs by count:")
print(df.head(25).to_markdown(index=False, numalign="left", stralign="left"))  # Nicer output

# Display the total number of unique TLDs
total_unique_tlds = len(df)
print(f"Total number of unique TLDs: {total_unique_tlds}")

# Display the top 25 TLDs by count
print("Top 25 TLDs by count:")
print(df.head(25).to_markdown(index=False, numalign="left", stralign="left"))

In [ ]:
# Query to fetch the top 25 TLDs by count
query = """
SELECT tld, COUNT(*) AS count
FROM DomainList
GROUP BY tld
ORDER BY count DESC
LIMIT 25
"""

# Execute the query and fetch the data
cursor.execute(query)
tld_data = cursor.fetchall()

# Create a pandas DataFrame from the fetched data
df = pd.DataFrame(tld_data)

# Visualize the TLD distribution using Seaborn
chart = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        x=alt.X("tld", sort="-y", axis=alt.Axis(labelAngle=-45)),
        y="count",
        tooltip=["tld", "count"],
    )
    .properties(title="Top TLDs by Count")
    .interactive()
)

chart

In [ ]:
# Query to fetch data
query = """
SELECT ds.id AS source_id, COUNT(dls.domain_list_id) AS domain_count, ds.organisation
FROM DomainListDomainSource dls
JOIN DomainSource ds ON dls.domain_source_id = ds.id
GROUP BY ds.id, ds.organisation
"""

df_sources = None

# Fetch data into a DataFrame
try:
    cursor.execute(query)
    data = cursor.fetchall()
    df_sources = pd.DataFrame(data, columns=["source_id", "domain_count", "organisation"])
except mysql.connector.Error as e:
    print(f"Error fetching data: {e}")

# If df_sources is successfully defined, proceed with analysis
if df_sources is not None:
    # Calculate relative distribution
    total_domains = df_sources["domain_count"].sum()
    df_sources["relative"] = df_sources["domain_count"] / total_domains * 100

    # Visualize the data
    plt.figure(figsize=(12, 6))
    sns.barplot(
        x="organisation",
        y="relative",
        data=df_sources,
        hue="organisation",
        palette="viridis",
        legend=False,
    )
    plt.title("Relative Distribution of Domains to Sources")
    plt.xlabel("Source Organisation")
    plt.ylabel("Relative Percentage (%)")
    plt.xticks(rotation=45)
    plt.show()

    # Display the DataFrame with relative percentages
    print("\nData with relative percentages:")
    print(df_sources)
else:
    print("No data fetched from the database.")

In [ ]:
# Format the domain count with dots for readability
df_sources["domain_count"] = df_sources["domain_count"].apply(lambda x: f"{x:,}".replace(",", "."))
# Sort by domain_count for better visualization
df_sources_sorted = df_sources.sort_values(by="domain_count", ascending=False)
# Visualize the absolute numbers
plt.figure(figsize=(14, 8))
sns.barplot(
    x="organisation",
    y=df_sources_sorted["domain_count"].str.replace(".", "").astype(int),
    hue="organisation",
    data=df_sources_sorted,
    palette="viridis",
    legend=False,
)
plt.title("Absolute Number of Domains per Source")
plt.xlabel("Source Organisation")
plt.ylabel("Number of Domains")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()
# Display the DataFrame with absolute domain counts
print("\nAbsolute Number of Domains per Source:")
print(df_sources_sorted)

In [ ]:
# Query to fetch data
query = """
SELECT ds.id AS source_id, COUNT(dls.domain_list_id) AS domain_count, ds.organisation
FROM DomainListDomainSource dls
JOIN DomainSource ds ON dls.domain_source_id = ds.id
GROUP BY ds.id, ds.organisation
"""

df_sources = None

# Fetch data into a DataFrame
try:
    cursor.execute(query)
    data = cursor.fetchall()
    df_sources = pd.DataFrame(data, columns=["source_id", "domain_count", "organisation"])
except mysql.connector.Error as e:
    print(f"Error fetching data: {e}")

# If df_sources is successfully defined, proceed with printing the list
if df_sources is not None:
    # Print list of sources with their total domain counts
    print("Total domains per source:")
    for index, row in df_sources.iterrows():
        formatted_count = "{:,}".format(row["domain_count"]).replace(",", ".")  # Format with dots
        print(f"{row['organisation']}: {formatted_count} domains")
else:
    print("No data fetched from the database.")